# Evaluation: validation -> settings, then test (issues #6-#8, #32)

1. On the **validation** set: choose the score method, the image threshold and
   the pixel threshold (best MCC), then freeze them.
2. On the **test** set: apply the frozen settings, **once**, and report the
   metrics. If anything is changed after looking at the test results, the test
   is no longer a clean measure.

Procedure in `docs/architecture_v0.1.md`. **Runtime: GPU.**

| Cell | What it does |
|---|---|
| 1 | Parameters (your values) |
| 2 | Mount Drive, get the code, install the packages |
| 3 | Copy the dataset to the VM, load the trained model |
| 4 | Validation: choose and save the settings |
| 5 | Test: metrics with the frozen settings, maps saved |
| 6 | Results per difficulty level / per scene |
| 7 | Gallery: best successes and most telling failures |
| 8 | Predict one patch: a test patch with its true mask by default, or any 512x512 .tif |

In [ ]:
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"

DATASET_ZIP = DRIVE_ROOT + "/data/dataset_v0.1_level1.zip"
LOCAL_DATASET = "/content/dataset"
RUN_NAME = "v0.1_level1_resnet34"
RUN_DIR = DRIVE_ROOT + "/runs/" + RUN_NAME
EVAL_DIR = RUN_DIR + "/eval"

In [ ]:
# Mount Google Drive, get the latest code from GitHub, install the packages.
# Everything happens on the Colab VM: nothing is installed on your PC.
import os
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/aletheia"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "-r", REPO_DIR + "/requirements.txt"], check=True)

# The notebook runs from its own folder and imports the module next to it;
# the repository root on sys.path makes `from features.<name>...` work too.
os.chdir(REPO_DIR + "/features/evaluation")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
import os

from features.sar_io.sar_io import unzip_folder
from features.unet.unet import load_model

if not os.path.exists(LOCAL_DATASET + "/manifest.csv"):
    unzip_folder(DATASET_ZIP, LOCAL_DATASET)
model, config = load_model(RUN_DIR)
print("Model loaded, commit", config.get("git_commit"))

In [ ]:
import json
from pathlib import Path

from evaluation import choose_thresholds, collect_statistics
from features.unet.unet import iterate_predictions

print("Validation set:")
val_stats = collect_statistics(iterate_predictions(model, config, LOCAL_DATASET, "val"))
settings = choose_thresholds(val_stats)

Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
Path(EVAL_DIR + "/settings.json").write_text(json.dumps(settings, indent=2))
config.update(settings)

In [ ]:
from evaluation import compute_metrics, print_summary
from features.sar_io.sar_io import read_manifest

print("Test set (frozen settings):")
test_stats = collect_statistics(iterate_predictions(model, config, LOCAL_DATASET, "test"),
                                save_maps_dir=EVAL_DIR + "/pred_maps")
manifest = read_manifest(LOCAL_DATASET + "/manifest.csv")
summary, results = compute_metrics(test_stats, settings, manifest)
print_summary(summary)

results.to_csv(EVAL_DIR + "/results_test.csv", index=False)
summary_to_save = {k: v for k, v in summary.items()}
Path(EVAL_DIR + "/summary_test.json").write_text(json.dumps(summary_to_save, indent=2))

In [ ]:
from evaluation import metrics_by_group

display(metrics_by_group(results, "level"))
display(metrics_by_group(results, "scene"))

In [ ]:
import matplotlib.pyplot as plt

from evaluation import plot_sample
from features.sar_io.sar_io import read_probability_map, read_tif


def show(row):
    patch = read_tif(f"{LOCAL_DATASET}/{row['file']}")
    mask = read_tif(f"{LOCAL_DATASET}/{row['mask_file']}") if row["label"] == 1 else None
    probability_map = read_probability_map(f"{EVAL_DIR}/pred_maps/{row['sample_id']}.tif")
    title = f"{row['sample_id']}  score {row['score']:.2f}"
    if row["label"] == 1:
        title += f"  pixel MCC {row['pixel_mcc']:.2f}"
    plot_sample(patch, mask, probability_map, title, config["input_scale"], settings["pixel_threshold"])
    plt.show()


forged = results[results["label"] == 1].sort_values("pixel_mcc")
print("Best forged patches:")
for _, row in forged.tail(3).iterrows():
    show(row)
print("Worst forged patches:")
for _, row in forged.head(3).iterrows():
    show(row)
print("Authentic patches with the highest score (false alarms):")
for _, row in results[results["label"] == 0].sort_values("score").tail(3).iterrows():
    show(row)

In [ ]:
# Predict ONE patch. By default: a random forged patch of the TEST set (never
# seen in training), shown with its true mask. Change `random_state` for
# another one; `real/` instead of `forged/` gives its authentic twin.
# Any 512x512 single-band .tif works too, e.g. the hand-made military-base
# example (#11): set PATCH_PATH to it and MASK_PATH = None.
import os

from features.sar_io.sar_io import read_manifest
from features.unet.unet import predict

manifest = read_manifest(LOCAL_DATASET + "/manifest.csv")
test_forged = manifest[(manifest["split"] == "test") & (manifest["label"] == 1)]
row = test_forged.sample(1, random_state=0).iloc[0]
PATCH_PATH = LOCAL_DATASET + "/" + row["file"]
MASK_PATH = LOCAL_DATASET + "/" + row["mask_file"]

# Uses the settings chosen on val (cell 4); `zone` = pixels called forged.
score, is_forged, probability_map, zone = predict(model, config, PATCH_PATH)
print(f"{os.path.basename(PATCH_PATH)}: score {score:.3f} -> {'FORGED' if is_forged else 'authentic'}, "
      f"zone: {zone.mean() * 100:.2f} % of the patch")
mask = read_tif(MASK_PATH) if MASK_PATH else None
fig = plot_sample(read_tif(PATCH_PATH), mask, probability_map, f"score {score:.2f}",
                  config["input_scale"], settings["pixel_threshold"])
plt.show()
# fig.savefig(EVAL_DIR + "/example.png", dpi=200)   # for the slides